# Code

In [ ]:
# is cuda available?
import torch
"cuda" if torch.cuda.is_available() else "cpu"

In [ ]:
import numpy as np
def create_balanced_split(dataset, val_samples_per_class=None):
    """
    Split dataset into train and validation sets with equal samples per class.
    If val_samples_per_class is None, uses 10% of the smallest class.
    """
    # Get all targets (class labels)
    targets = np.array(dataset.targets)
    
    # Find the size of each class
    class_sizes = []
    for class_idx in range(len(dataset.classes)):
        class_indices = np.where(targets == class_idx)[0]
        class_sizes.append(len(class_indices))
    
    # If not specified, take 10% of the smallest class
    if val_samples_per_class is None:
        val_samples_per_class = int(min(class_sizes) * 0.1)
    
    print(f"\nTaking {val_samples_per_class} samples per class for validation")
    
    train_indices = []
    val_indices = []
    
    # For each class, take equal number of samples
    for class_idx in range(len(dataset.classes)):
        # Get all indices for this class
        class_indices = np.where(targets == class_idx)[0]
        
        # Shuffle indices for this class
        np.random.shuffle(class_indices)
        
        # Take fixed number for validation
        val_indices.extend(class_indices[:val_samples_per_class])
        train_indices.extend(class_indices[val_samples_per_class:])
    
    return train_indices, val_indices

In [ ]:
import torch
import torchvision.models as models
import torchvision.transforms as transforms
from torch.utils.data import DataLoader, Subset
from torchvision.datasets import ImageFolder
import torch.nn as nn
import torch.optim as optim
import numpy as np

# Define data transformations for grayscale images
transform = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),  # Convert to grayscale
    transforms.Resize((48, 48)),  # Resize to 48x48
    transforms.ToTensor(),  # Convert to tensor
    transforms.Normalize(mean=[0.5], std=[0.5])  # Normalize for 1 channel
])

# Load the full dataset from the folder structure
full_dataset = ImageFolder(root='train', transform=transform)

# Set random seed for reproducibility
np.random.seed(42)

train_indices, val_indices = create_balanced_split(full_dataset, 300)

train_dataset = Subset(full_dataset, train_indices)
val_dataset = Subset(full_dataset, val_indices)

train_loader = DataLoader(
    train_dataset,
    batch_size=32,
    shuffle=True,
    num_workers=4
)

valid_loader = DataLoader(
    val_dataset,
    batch_size=32,
    shuffle=False,
    num_workers=4
)

# Print dataset information
print(f"Number of classes: {len(full_dataset.classes)}")
print(f"Classes: {full_dataset.classes}")
print(f"Total images: {len(full_dataset)}")
print(f"Training images: {len(train_dataset)}")
print(f"Validation images: {len(val_dataset)}")
print(f"Class to index mapping: {full_dataset.class_to_idx}")

# Verify balanced split
print("\nClass distribution in validation set:")
val_targets = [full_dataset.targets[i] for i in val_indices]
for class_idx, class_name in enumerate(full_dataset.classes):
    count = val_targets.count(class_idx)
    print(f"  {class_name}: {count} images")

# Load ResNet50 and modify for grayscale input and 7 classes
resnet50_model = models.resnet50(pretrained=True)

# Modify first layer for grayscale (1 channel)
pretrained_weight = resnet50_model.conv1.weight.data
averaged_weight = pretrained_weight.mean(dim=1, keepdim=True)
resnet50_model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
resnet50_model.conv1.weight.data = averaged_weight

# Modify final layer for 7 classes
num_classes = 7
resnet50_model.fc = nn.Linear(resnet50_model.fc.in_features, num_classes)

# Move model to GPU if available
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
resnet50_model = resnet50_model.to(device)

print(f"\nModel moved to: {device}")

In [ ]:
import matplotlib.pyplot as plt
def metrics(num_epochs, train_losses, valid_losses, train_accuracies, valid_accuracies):
    plt.figure(figsize=(12, 5))
    plt.subplot(1, 2, 1)
    plt.plot(range(1, num_epochs + 1), train_losses, label='Train Loss')
    plt.plot(range(1, num_epochs + 1), valid_losses, label='Valid Loss')
    plt.xlabel('Epochs')
    plt.ylabel('Loss')
    plt.legend()
    plt.title('Loss overtime')

    plt.subplot(1, 2, 2)
    plt.plot(range(1, num_epochs + 1), train_accuracies, label='Train Accuracy')
    plt.plot(range(1, num_epochs + 1), valid_accuracies, label='Valid Accuracy')
    plt.xlabel('Epochs')
    plt.ylabel('Accuracy (%)')
    plt.legend()
    plt.title('Accuracy overtime')
    plt.savefig("metrics.png", bbox_inches='tight')
    plt.show()

In [ ]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(filter(lambda p: p.requires_grad, resnet50_model.parameters()))

# Training loop
def train_model(model, train_loader, valid_loader, criterion, optimizer, num_epochs=10, learning_rate= 0.0001):
    print("start training using ")
    print(device)
#     backbone_params = [p for name, p in model.named_parameters() if 'fc' not in name]
#     classifier_params = [p for name, p in model.named_parameters() if 'fc' in name]

#     optimizer = torch.optim.Adam([
#         {'params': backbone_params, 'lr': learning_rate * 0.1},  # Lower for pretrained
#         {'params': classifier_params, 'lr': learning_rate}       # Higher for new layer
# ])
    train_losses, valid_losses = [], []
    best_valid_loss = 1000
    train_accuracies, valid_accuracies = [], []
    
    for epoch in range(num_epochs):
        model.train()
        running_loss, correct, total = 0.0, 0, 0
        # device = next(model.parameters()).device
        for i, (images, labels) in enumerate(train_loader):
            # print(images.dtype, device)
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()
            
            running_loss += loss.item()
            _, predicted = outputs.max(1)
            correct += (predicted == labels).sum().item()
            total += labels.size(0)
            # if (i + 1) % 10 == 0:
            #     print(f'Epoch [{epoch+1}/{num_epochs}], Step [{i+1}/{len(train_loader)}], '
            #           f'Loss: {loss.item():.4f}')
        
        train_loss = running_loss / len(train_loader)
        train_accuracy = 100 * correct / total
        train_losses.append(train_loss)
        train_accuracies.append(train_accuracy)

        # Évaluation sur valid_loader
        model.eval()
        running_loss, correct, total = 0.0, 0, 0
        with torch.no_grad():
            for images, labels in valid_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)
                running_loss += loss.item()
                _, predicted = outputs.max(1)
                correct += (predicted == labels).sum().item()
                total += labels.size(0)

        valid_loss = running_loss / len(valid_loader)
        valid_accuracy = 100 * correct / total
        valid_losses.append(valid_loss)
        valid_accuracies.append(valid_accuracy)
 
        print(f"Epoch {epoch+1}: Train Loss: {train_loss:.4f}, Train Acc: {train_accuracy:.2f}%, Valid Loss: {valid_loss:.4f}, Valid Acc: {valid_accuracy:.2f}%")
        if best_valid_loss > valid_loss:
            torch.save(model.state_dict(), "emotion_resnet18_best.pth")
            print(f"Saved best new validalition loss with a value of {valid_loss}")
            best_valid_loss = valid_loss
            
    print('Finished Training')
    torch.save(model.state_dict(), 'emotion_resnet18_final.pth')
    print('Model saved as emotion_resnet18.pth')
    metrics(num_epochs, train_losses, valid_losses, train_accuracies, valid_accuracies)

In [ ]:
train_model(resnet50_model,train_loader,valid_loader, criterion, optimizer, 10, learning_rate=0.0001)

In [ ]:
# Load the saved model
model = models.resnet18(pretrained=False)

# Modify architecture (same as training)
model.conv1 = nn.Conv2d(1, 64, kernel_size=7, stride=2, padding=3, bias=False)
model.fc = nn.Linear(model.fc.in_features, 7)

# Load trained weights
model.load_state_dict(torch.load('emotion_resnet18.pth'))
model.eval()